(sec:T3:SF:interpretacion)=
### Interpretación espectral de la serie de Fourier

Desde un punto de vista espectral, la serie de Fourier proporciona una representación discreta del contenido en frecuencia de una señal periódica. Cada coeficiente $c_k$ está asociado a una componente armónica de frecuencia $k\omega_0$, de modo que la periodicidad en el dominio temporal se traduce en una discretización del espectro en el dominio de la frecuencia. Como ejemplo, se muestra en la figura la serie de Fourier correspondiente a un tren de pulsos triangulares.

```{figure} figures/T3/signal_tri_periodic2.svg
:name: figs:T3:SF:interpretacion_tiempo
:width: 80%
:alt: Tren de pulsos triangulares.
:align: center
```
```{figure} figures/T3/signal_tri_periodic2_dk.svg
:name: figs:T3:SF:interpretacion_frecuencia
:width: 80%
:alt: Serie de Fourier correspondiente a un tren de pulsos triangulares.
:align: center
```

\begin{figure}[h]
	\centering
	\includegraphics[width=\linewidth]{figs/SF}
	\caption{Serie de Fourier correspondiente a un tren de pulsos triangulares.}
	\label{fig:SF}
\end{figure}

Esta interpretación permite analizar de forma cualitativa el comportamiento de las señales y los sistemas:
- Señales cuyos coeficientes son significativos únicamente para valores pequeños de $|k|$ presentan una variación lenta en el tiempo.
- La presencia de armónicos de orden elevado se asocia a variaciones rápidas o a la existencia de discontinuidades.

:::{tip} Interpretación espectral
En la siguiente gráfica interactiva se analiza la relación entre la variación temporal de la señal $\tilde{x}(t)$ y el módulo de los coeficientes de su serie de Fourier, $|c_k|$, siendo:
```{math}
    \tilde{x}(t)=\tanh(s \cdot \sin(\omega_0 t))
```
Se puede variar la Abrupticidad ($s$), transformando la señal desde una senoide pura hasta una onda cuadrada.

Observa:
- Suavidad vs. Ancho de banda: Cuando la señal es suave (valores bajos de abrupticidad), varía lentamente y su energía se concentra en frecuencias bajas (solo $k=\pm 1$).
- Discontinuidades: Al aumentar la abrupticidad, las transiciones se vuelven rápidas y violentas. Para construir estos cambios bruscos, es necesario que aparezcan armónicos de orden elevado (el espectro se ensancha).
- Potencia media: Al hacer la señal más cuadrada, notarás que los coeficientes fundamentales ($k=\pm 1$) también crecen. Esto ocurre porque la señal "se ensancha" en el tiempo, aumentando su área y potencia media[^foot1].
:::

[^foot1]: Ver {ref}`subsec:T3:SF:parseval`.

In [56]:
import numpy as np
from bokeh.plotting import figure, show
from bokeh.layouts import column, row
from bokeh.models import ColumnDataSource, CustomJS, Slider, Div
from bokeh.io import output_notebook

# Importamos las funciones auxiliares
from utils.plot_helpers import style_math_axes, add_math_ticks

output_notebook()

# ==========================================
# 1. PARÁMETROS INICIALES
# ==========================================
VAL_STEEPNESS = 0.5  
N_POINTS = 400       # Resolución temporal
N_HARMONICS = 20     # Armónicos a cada lado (+/-)

# Colores definidos para usar en gráficas y etiquetas HTML
COLOR_TIME = "blue" # Azul Bokeh
COLOR_FREQ = "blue" # Rojo Bokeh

t_min, t_max = -2.5, 2.5
t_vals = np.linspace(t_min, t_max, N_POINTS)

# ==========================================
# 2. GENERACIÓN DE DATOS
# ==========================================

# A. Datos Tiempo
def generate_signal(t_arr, s):
    if s < 1e-3: s = 1e-3
    # Señal normalizada
    norm = np.tanh(s)
    return np.tanh(s * np.sin(np.pi * t_arr)) / norm

y_vals = generate_signal(t_vals, VAL_STEEPNESS)
source_time = ColumnDataSource(data=dict(t=t_vals, y=y_vals))

# B. Datos Frecuencia (Simétrico)
# Rango de k: -20 ... 0 ... +20
k_vals = np.arange(-N_HARMONICS, N_HARMONICS + 1)
t_period = np.linspace(-1, 1, N_POINTS, endpoint=False) # Un periodo para integrar

def calc_dft(s):
    if s < 1e-3: s = 1e-3
    norm = np.tanh(s)
    y_p = np.tanh(s * np.sin(np.pi * t_period)) / norm
    
    mags = []
    for k in k_vals:
        # Proyección (Integral numérica simple)
        # e^(-j w k t) = cos(...) - j sin(...)
        angle = k * np.pi * t_period
        basis_real = np.cos(angle)
        basis_imag = -np.sin(angle)
        
        re = np.mean(y_p * basis_real)
        im = np.mean(y_p * basis_imag)
        
        mag = np.sqrt(re**2 + im**2)
        # Limpieza de valores muy pequeños (ruido numérico)
        if mag < 1e-4: mag = 0
        mags.append(mag)
        
    return np.array(mags)

mag_vals = calc_dft(VAL_STEEPNESS)
source_freq = ColumnDataSource(data=dict(k=k_vals, mag=mag_vals, zeros=np.zeros_like(k_vals)))


# ==========================================
# 3. GRÁFICOS
# ==========================================

# --- TIEMPO ---
# Sin título, sin leyenda
p_time = figure(width=600, height=300) 
style_math_axes(p_time, x_range=(t_min, t_max), y_range=(-1.3, 1.3), xlabel="t", ylabel=r"$$\tilde{x}(t)$$")
add_math_ticks(p_time, yticks=[-1, 1], ytick_labels=["-1", "1"], tick_len=5)

p_time.line('t', 'y', source=source_time, color=COLOR_TIME, line_width=3)

# --- FRECUENCIA ---
# Rango simétrico en X
p_freq = figure(width=600, height=300)
style_math_axes(p_freq, x_range=(-N_HARMONICS-1, N_HARMONICS+1), y_range=(0, 0.7), xlabel="k", ylabel=r"$$|c_k|$$")

p_freq.segment(x0='k', y0='zeros', x1='k', y1='mag', source=source_freq, color=COLOR_FREQ, line_width=3)
p_freq.scatter('k', 'mag', source=source_freq, color=COLOR_FREQ, size=8, marker="circle")


# ==========================================
# 4. INTERACTIVIDAD
# ==========================================
s_steep = Slider(start=0.1, end=10.0, value=VAL_STEEPNESS, step=0.1, title=r"Abrupticidad ($$s$$)")

callback = CustomJS(
    args=dict(source_t=source_time, source_f=source_freq, s_steep=s_steep),
    code="""
    const s = s_steep.value;
    const PI = Math.PI;
    
    // --- 1. TIEMPO ---
    const t = source_t.data['t'];
    const y = source_t.data['y'];
    let norm = Math.tanh(s);
    if (Math.abs(norm) < 1e-9) norm = 1.0;

    for (let i = 0; i < t.length; i++) {
        y[i] = Math.tanh(s * Math.sin(PI * t[i])) / norm;
    }
    source_t.change.emit();

    // --- 2. FRECUENCIA (DFT Bilateral) ---
    const N_integ = 200; 
    const k_arr = source_f.data['k'];
    const mag = source_f.data['mag'];
    
    // Generamos un periodo de señal auxiliar para integrar
    let y_p = new Float32Array(N_integ);
    let t_p = new Float32Array(N_integ);
    for(let i=0; i<N_integ; i++){
        // t va de -1 a 1
        let ti = -1 + (2 * i / N_integ);
        t_p[i] = ti;
        y_p[i] = Math.tanh(s * Math.sin(PI * ti)) / norm;
    }
    
    for (let i = 0; i < k_arr.length; i++) {
        let k = k_arr[i];
        let sum_re = 0.0;
        let sum_im = 0.0;
        
        for (let j = 0; j < N_integ; j++) {
            let angle = PI * k * t_p[j]; // w0 = pi
            // exp(-j...) = cos - j*sin
            sum_re += y_p[j] * Math.cos(angle);
            sum_im += y_p[j] * (-Math.sin(angle));
        }
        
        let re = sum_re / N_integ;
        let im = sum_im / N_integ;
        let m = Math.sqrt(re*re + im*im);
        
        if (m < 1e-3) m = 0;
        mag[i] = m;
    }
    source_f.change.emit();
""")

s_steep.js_on_change('value', callback)

# ==========================================
# 5. LAYOUT
# ==========================================

# Etiquetas de Color (Leyenda estática arriba)
# header_html = f"""
# <div style="display: flex; justify-content: center; gap: 40px; font-family: sans-serif; margin-bottom: 10px; font-size: 16px;">
#     <div style="color: {COLOR_TIME}; font-weight: bold;">
#         &#9644; Señal Temporal x(t)
#     </div>
#     <div style="color: {COLOR_FREQ}; font-weight: bold;">
#         &#9679; Magnitud Espectral |c<sub>k</sub>|
#     </div>
# </div>
# """
# header = Div(text=header_html)

caption_text = """
<div style="font-family: sans-serif; margin-top: 15px; font-size: 14px; color: #444;">
    <p><b>Interpretación espectral:</b> suavidad/abrupticidad y ancho de banda.</p>
</div>
"""
caption = Div(text=caption_text)

layout = column(s_steep, p_time, p_freq, caption, sizing_mode="scale_width")
show(layout)

Loading BokehJS ...

Por otro lado, teniendo en cuenta que las exponenciales complejas son autofunciones de los sistemas LTI, el desarrollo en serie de Fourier a la salida de un sistema LTI:

```{figure} figures/T3/diag4.svg
:name: figs:T3:SF:LTI_SF_salida
:width: 80%
:alt: Serie de Fourier correspondiente a un tren de pulsos triangulares.
:align: center
```

Tendrá como coeficientes:
```{math}
	\tilde{y}(t) \SF d_k = H(k\omega_0)\,c_k
```
donde $H(k\omega_0)$ es la respuesta en frecuencia del sistema a las frecuencias armónicas de la fundamental.

Asimismo, esta representación discreta del espectro constituye el punto de partida para establecer relaciones estructurales con otros conceptos del análisis de Fourier, como la transformada de Fourier de señales aperiódicas y la interpretación del muestreo como fenómeno dual de la periodicidad.